# Updated source — *How Much Preprocessing Is Enough?*

Everything that changed in the project's code **after** the original benchmark
run, consolidated into one notebook.

## Why this notebook exists

Writing the paper produced one genuine bug fix and four new analyses. Rather than
leave them scattered across patches, they are collected here in the order they
arose, each with the reason it was needed.

| # | Change | Kind | Consequence for the paper |
|---|---|---|---|
| 1 | Ordinal solver received non-contiguous labels | **Bug fix** | RQ4 changed from "inconclusive" to a split verdict: the ordinal model is refuted, retrieval succeeds |
| 2 | Portfolio-composition sensitivity | New analysis | The headline magnitude was shown to be fragile; the claim was weakened from a fixed ratio to a range |
| 3 | ROC-AUC vs balanced accuracy at the weighting rung | New analysis | A reported null became a documented metric artefact |
| 4 | Diagnosis of the three empty datasets | New analysis | "Unexplained failure" became two named, recoverable causes |
| 5 | Tolerance sweep and reliable-label subset | New analysis | Two robustness layers the Methods promised but the run never delivered |

## What this notebook does *not* do

It does not re-run the 9,405-cell factorial. That remains in `PreProcessing.ipynb`,
which loads the datasets, fits every pipeline and writes `raw_results.csv`. This
notebook starts from that output, so it runs in a couple of minutes on a laptop
with no GPU.

## Layout

| Part | Contents |
|---|---|
| **0** | Setup and shared helpers |
| **1** | **The patch** — what broke, why, and the drop-in replacement |
| **2** | Re-run of the predictive analysis with the patch applied |
| **3** | The four new analyses |
| **4** | Reproduction of every manuscript table |
| **5** | **Verification** — 273 assertions over data, manuscript and arithmetic |

## Inputs

`raw_results.csv`, `table1_dataset_characteristics.csv`, `tuned_results.csv`,
`results_seed7.csv`, `results_seed13.csv`, `table4_pivot.csv`,
`variance_stability_5rep.csv`, `multiseed_delta_eps2.csv`,
`table8_loo_sensitivity.csv`, `moderator_data.csv`, `manuscript.tex`

# Part 0 — Setup

## 0.1 Environment and data

In [ ]:
# pip install pandas numpy scipy statsmodels scikit-learn mord
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, spearmanr, levene, chi2
from statsmodels.stats.multitest import multipletests

# resolve the data directory whether the notebook is run from notebooks/,
# from the repository root, or from a directory holding the CSVs directly
def _find_dir(marker, *cands):
    for c in cands:
        if (Path(c) / marker).exists():
            return Path(c).resolve()
    raise FileNotFoundError(f"{marker} not found in {cands}")

DATA   = _find_dir("raw_results.csv", ".", "../data", "data", "../../data")
TABLES = _find_dir("table4_pivot.csv", ".", "../tables", "tables", str(DATA))
print(f"data   : {DATA}")
print(f"tables : {TABLES}")          # directory holding the released CSVs
OUT  = Path("outputs"); OUT.mkdir(exist_ok=True)

LEVELS = ["P1","P2","P3","P4","P5","P6","P7","P8a","P8b"]
K      = len(LEVELS)
SEEDS  = [42, 7, 13]
TAU    = 0.01

pd.set_option("display.width", 120, "display.max_columns", 40)

raw = pd.read_csv(DATA/"raw_results.csv")
meta = pd.read_csv(DATA/"table1_dataset_characteristics.csv").set_index("dataset")
ok  = raw[raw.fold_mean.notna()].copy()
ok["task"] = ok.dataset.map(meta.task)
ok["IR"]   = ok.dataset.map(meta.imbalance_ratio)

print(f"scheduled cells {len(raw):,} | valid {len(ok):,} ({100*len(ok)/len(raw):.2f}%)")
print(f"datasets loaded {raw.dataset.nunique()} | contributing {ok.dataset.nunique()}")

## 0.2 Shared helpers

In [ ]:
def kendall_w(piv):
    """Kendall's W = rank-based epsilon^2 for a datasets x levels score matrix."""
    piv = piv.dropna()
    n, k = piv.shape
    ranks = piv.rank(axis=1, ascending=False).values
    chi2_f = 12*n/(k*(k+1)) * np.sum((ranks.mean(0) - (k+1)/2)**2)
    return chi2_f/(n*(k-1)), chi2_f, n, k

def iman_davenport(chi2_f, n, k):
    denom = n*(k-1) - chi2_f
    return np.inf if denom <= 0 else (n-1)*chi2_f/denom

def prep_pivot(df):
    return df.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")[LEVELS]

def algo_pivot(df, exclude=("A10_TabPFN",)):
    d = df[~df.algo_id.isin(exclude)]
    return d.groupby(["dataset","algo_id"])["fold_mean"].mean().unstack("algo_id")

def bootstrap_delta(pp, pa, B=10_000, seed=42):
    """Dataset-level bootstrap of Delta eps^2 = W_prep - W_alg."""
    ds = sorted(set(pp.dropna().index) & set(pa.dropna().index))
    pp, pa = pp.loc[ds], pa.loc[ds]
    point = kendall_w(pp)[0] - kendall_w(pa)[0]
    rng = np.random.RandomState(seed)
    reps = np.empty(B)
    for b in range(B):
        s = rng.choice(len(ds), len(ds), replace=True)
        reps[b] = kendall_w(pp.iloc[s])[0] - kendall_w(pa.iloc[s])[0]
    lo, hi = np.percentile(reps, [2.5, 97.5])
    return point, lo, hi, len(ds), (reps > 0).mean()

# Part 1 — The patch

## 1.1 What broke

The predictive-MSP analysis in the original run (`Cell 8d`) reported the
meta-model's predictions as identical to baseline $B_1$ on all 2,889 instances,
with no confidence values at all. That is not a result; it is a silent failure.

The cause is a requirement of the solver that the label set did not satisfy.
`mord.LogisticAT` expects the training labels to occupy a **contiguous** range
$\{0,\dots,C-1\}$. Minimal-sufficiency labels are a *subset* of the nine ladder
rungs, with gaps wherever a level is never minimal — in this suite $P_5$ never is,
and $P_{8\mathrm{a}}$ appears in 0.1 % of instances. So the solver raised:

```
ValueError: Values in y must be [0 1 2 3 4 5 6 7], instead got [0 1 2 3 5 6 7 8]
```

and the surrounding `try/except` caught it and fell through to the baseline.

## 1.2 Why it went unnoticed

The fallback was designed for a *missing dependency*, not a *rejected input*. It
caught a broad `Exception`, substituted a baseline, and recorded the substitution
in a field nobody inspected. The run completed, the tables populated, and the
numbers looked plausible because a strong baseline is, by construction, plausible.

The lesson generalises: a fallback that produces output indistinguishable from a
legitimate result is worse than a crash.

## 1.3 Confirming the diagnosis

The cell below reproduces the original failure, then shows the fix working on the
same data.

In [ ]:

from mord import LogisticAT
from sklearn.preprocessing import StandardScaler
import numpy as np

# a label set with the same gap structure as the real MSP labels
y_gapped = np.array([0,0,0,1,1,3,3,5,6,8] * 12)
X_demo   = np.random.RandomState(0).randn(len(y_gapped), 4)

print("label set:", sorted(set(y_gapped)), " -> gaps at 2, 4, 7\n")

print("BEFORE the patch")
try:
    LogisticAT(alpha=1.0).fit(X_demo, y_gapped)
    print("   fitted (unexpected)")
except Exception as e:
    print(f"   {type(e).__name__}: {e}")
    print("   -> the original code caught this and returned the B1 baseline")

print("\nAFTER the patch")
classes = np.unique(y_gapped)
enc = {c: i for i, c in enumerate(classes)}
mdl = LogisticAT(alpha=1.0).fit(X_demo, np.array([enc[v] for v in y_gapped]))
decoded = classes[np.clip(mdl.predict(X_demo).astype(int), 0, len(classes)-1)]
proba = mdl.predict_proba(X_demo)
print(f"   fitted; predictions decoded back to {sorted(set(decoded))}")
print(f"   predict_proba shape {proba.shape}, confidences in "
      f"[{proba.max(axis=1).min():.3f}, {proba.max(axis=1).max():.3f}]")


## 1.4 The drop-in replacement

Replace the model-fitting block of `Cell 8d` in `PreProcessing.ipynb` with this
function. Three changes from the original:

1. labels are encoded to a contiguous range before fitting and decoded after;
2. `predict_proba` is called so the abstention layer has confidences to work with;
3. the exception handler **re-raises** instead of substituting a baseline, so a
   future failure of this kind stops the run rather than disguising itself.

In [ ]:

def fit_ordinal_msp(X_train, y_train, X_test, alpha=1.0, n_levels=9):
    """Fit the all-threshold ordinal model and predict, handling gapped labels.

    Drop-in replacement for the model block of Cell 8d.

    Returns
    -------
    pred : ndarray of int, in the ORIGINAL label space
    conf : ndarray of float, max posterior probability per instance

    Raises
    ------
    RuntimeError
        If the model cannot be fitted. Deliberately fatal: the original code
        fell back to a baseline here, which produced output indistinguishable
        from a real result.
    """
    classes = np.unique(y_train)
    if len(classes) < 2:
        raise RuntimeError(f"ordinal fit needs >= 2 classes, got {classes}")

    # the fix: mord.LogisticAT requires labels in a contiguous range 0..C-1,
    # whereas MSP labels are a gapped subset of the nine ladder rungs
    enc = {c: i for i, c in enumerate(classes)}
    y_enc = np.array([enc[v] for v in y_train])

    try:
        model = LogisticAT(alpha=alpha).fit(X_train, y_enc)
        idx = np.clip(model.predict(X_test).astype(int), 0, len(classes) - 1)
        pred = classes[idx]
        conf = model.predict_proba(X_test).max(axis=1)
    except Exception as exc:
        raise RuntimeError(
            f"ordinal fit failed on labels {classes.tolist()}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc

    if not np.isfinite(conf).all():
        raise RuntimeError("model returned non-finite confidences")
    return np.clip(pred, 0, n_levels - 1), conf


print("fit_ordinal_msp ready — paste this into Cell 8d of PreProcessing.ipynb")


# Part 2 — Predictive MSP re-run (RQ4)

With the patch in place, the analysis produces a real result. The instance set is
identical to the original run — 963 instances over 92 datasets, base rate 0.7092 —
so the numbers are directly comparable; only the model changed.

In [ ]:
from mord import LogisticAT
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

SENS = {"A2_DecTree":0,"A3_RandForest":0,"A4_GradBoosting_SKL":0,"A6_XGBoost":0,
        "A7_LightGBM":0,"A8_CatBoost":0,"A10_TabPFN":1,"A1_Ridge_LogReg":2,
        "A9_MLP":3,"A5_SVM_SVR":4,"A11_kNN":4}
FEATS = ["log10_n","p","log10_np","miss","cat_density","has_hicard",
         "ir","n_classes","is_cls","sens"]

def build_instances(tau=TAU):
    rows = []
    for (d_, a), g in ok.groupby(["dataset","algo_id"]):
        s = g.set_index("pipeline")["fold_mean"].reindex(LEVELS)
        if s.notna().sum() == 0: continue
        m = meta.loc[d_]
        rows.append(dict(dataset=d_, algo_id=a,
            true_idx=int(np.argmax(((s >= s.max()-tau) & s.notna()).values)),
            log10_n=np.log10(m.n_samples), p=m.n_features,
            log10_np=np.log10(m.n_samples/max(m.n_features,1)),
            miss=m.miss_rate_pct, cat_density=m.n_categorical/max(m.n_features,1),
            has_hicard=float(m.n_high_card_cat>0), ir=m.imbalance_ratio,
            n_classes=m.n_classes, is_cls=float(m.task=="classification"),
            sens=SENS[a], best=s.max(), **{f"s_{L}": s[L] for L in LEVELS}))
    return pd.DataFrame(rows)

inst = build_instances()
base_rate = inst.true_idx.value_counts(normalize=True).max()
print(f"instances {len(inst)} | datasets {inst.dataset.nunique()} | base rate {base_rate:.4f}")

In [ ]:
def gap(row, p):
    """Performance forgone; if the rung is unavailable, climb to the next feasible one."""
    for j in range(p, K):
        if pd.notna(row[f"s_{LEVELS[j]}"]):
            return float(row.best - row[f"s_{LEVELS[j]}"])
    return np.nan

def lodo(inst, seed):
    inst = inst.copy()
    inst["regime"] = list(zip(inst.log10_n >= np.log10(5000),
                              inst.miss > 5, inst.ir >= 5))
    out = []
    for held in sorted(inst.dataset.unique()):
        tr, te = inst[inst.dataset != held], inst[inst.dataset == held]
        if tr.dataset.nunique() < 3: continue
        sc = StandardScaler().fit(tr[FEATS].values)
        Xtr, Xte, ytr = sc.transform(tr[FEATS]), sc.transform(te[FEATS]), tr.true_idx.values

        # ---- THE FIX: contiguous label encoding -------------------------------
        classes = np.unique(ytr)
        enc = {c: i for i, c in enumerate(classes)}
        mdl = LogisticAT(alpha=1.0).fit(Xtr, np.array([enc[v] for v in ytr]))
        pred = classes[np.clip(mdl.predict(Xte).astype(int), 0, len(classes)-1)]
        conf = mdl.predict_proba(Xte).max(axis=1)
        # -----------------------------------------------------------------------

        preds = {"P-MSP_meta": (pred, conf),
                 "B0_global":  (np.full(len(te), int(np.median(ytr))), None),
                 "B4_oracle":  (te.true_idx.values, None)}
        m1 = tr.groupby("algo_id").true_idx.agg(lambda s: int(s.value_counts().sort_index().idxmax()))
        preds["B1_algo_mode"] = (te.algo_id.map(m1).fillna(int(np.median(ytr))).astype(int).values, None)
        m2 = tr.groupby("regime").true_idx.agg(lambda s: int(s.value_counts().sort_index().idxmax()))
        preds["B2_regime"] = (te.regime.map(m2).fillna(int(np.median(ytr))).astype(int).values, None)
        nn = NearestNeighbors(n_neighbors=min(25, len(tr))).fit(Xtr)
        idx = nn.kneighbors(Xte, return_distance=False)
        preds["B3_meta_knn"] = (np.array([int(pd.Series(ytr[i]).value_counts().sort_index().idxmax())
                                          for i in idx]), None)

        for meth, (pr, cf) in preds.items():
            for j, (_, row) in enumerate(te.iterrows()):
                p = int(pr[j])
                out.append(dict(seed=seed, method=meth, dataset=held, algo_id=row.algo_id,
                                true_idx=int(row.true_idx), pred_idx=p,
                                exact=int(p == row.true_idx),
                                off_by_one=int(abs(p-row.true_idx) <= 1),
                                abs_err=abs(p-row.true_idx),
                                confidence=np.nan if cf is None else float(cf[j]),
                                perf_gap=gap(row, p), fit_fraction=(p+1)/K))
    return pd.DataFrame(out)

pred = pd.concat([lodo(inst, s) for s in SEEDS], ignore_index=True)
summary = (pred.groupby("method")
             .agg(exact=("exact","mean"), off_by_one=("off_by_one","mean"),
                  mae=("abs_err","mean"), gap=("perf_gap","mean"),
                  fit_fraction=("fit_fraction","mean")).round(4))
display(summary)
print(f"finite confidences: {pred[pred.method=='P-MSP_meta'].confidence.notna().sum()} "
      f"of {(pred.method=='P-MSP_meta').sum()}  (0 in the original run)")

In [ ]:
# McNemar: P-MSP against each baseline, and the strongest baseline against B1
piv = pred.pivot_table(index=["seed","dataset","algo_id"], columns="method", values="exact")
def mcnemar(a, b):
    n01 = int(((piv[a]==1) & (piv[b]==0)).sum()); n10 = int(((piv[a]==0) & (piv[b]==1)).sum())
    stat = 0.0 if n01+n10 == 0 else (abs(n01-n10)-1)**2/(n01+n10)
    return dict(comparison=f"{a} vs {b}", wins=n01, base_wins=n10,
                chi2=round(stat,2), p=1.0 if n01+n10==0 else float(chi2.sf(stat,1)))
rows = [mcnemar("P-MSP_meta", b) for b in
        ["B0_global","B1_algo_mode","B2_regime","B3_meta_knn","B4_oracle"]]
rows.append(mcnemar("B3_meta_knn","B1_algo_mode"))
display(pd.DataFrame(rows))

In [ ]:
# confidence-aware abstention: answer above threshold, else fall back to P8b
m = pred[(pred.method=="P-MSP_meta") & pred.confidence.notna()].copy()
term = inst.set_index(["dataset","algo_id"]).apply(lambda r: gap(r, K-1), axis=1)
m["term_gap"] = pd.Series(list(zip(m.dataset, m.algo_id)), index=m.index).map(term)
rows = []
for t in [.30,.40,.50,.60,.70,.80,.90,.95]:
    ans = (m.confidence >= t).values
    rows.append(dict(threshold=t, coverage=ans.mean(),
                     acc_on_covered=m.exact[ans].mean() if ans.any() else np.nan,
                     mean_gap=np.nanmean(np.where(ans, m.perf_gap, m.term_gap)),
                     mean_fit=np.mean(np.where(ans, m.fit_fraction, 1.0))))
display(pd.DataFrame(rows).round(4))

## 2.1 Robustness layers the Methods promised

Two layers were specified in the Methods but never delivered by the original run,
because a meta-model that always returns the baseline makes them vacuous. Both run
now: a sweep over the tolerance $\tau$, and a re-evaluation restricted to the
instances whose label is identical under all three cross-validation seeds.

In [ ]:
# robustness: tolerance sweep and reliable-label subset
for t in (0.005, 0.01, 0.02):
    i2 = build_instances(t)
    p2 = pd.concat([lodo(i2, s) for s in SEEDS], ignore_index=True)
    g = p2.groupby("method").exact.mean()
    print(f"tau={t}: base={i2.true_idx.value_counts(normalize=True).max():.4f}  "
          f"P-MSP={g['P-MSP_meta']:.4f}  B1={g['B1_algo_mode']:.4f}  B3={g['B3_meta_knn']:.4f}")

def labels_from(fname):
    r = pd.read_csv(DATA/fname); r = r[r.fold_mean.notna()]
    return {(d_,a): int(np.argmax(((s := g.set_index("pipeline")["fold_mean"]
              .reindex(LEVELS)) >= s.max()-TAU) & s.notna()))
            for (d_,a), g in r.groupby(["dataset","algo_id"])
            if g.fold_mean.notna().sum() > 0}
l = [labels_from(f) for f in ["raw_results.csv","results_seed7.csv","results_seed13.csv"]]
common = set(l[0]) & set(l[1]) & set(l[2])
stable = {k for k in common if l[0][k]==l[1][k]==l[2][k]}
print(f"\nseed-consistent labels: {len(stable)}/{len(common)} ({100*len(stable)/len(common):.1f}%)")
sub = pred[pd.Series(list(zip(pred.dataset,pred.algo_id)), index=pred.index).isin(stable)]
display(sub.groupby("method").agg(exact=("exact","mean"), gap=("perf_gap","mean")).round(4))

## 2.2 What changed in the paper

| Quantity | Original run | After the patch |
|---|---|---|
| P-MSP exact accuracy | 0.7092 (= $B_1$) | **0.6490** |
| $B_3$ meta-$k$NN exact | 0.7103 | **0.7799** |
| Finite confidences | 0 of 2,889 | **2,889 of 2,889** |
| Abstention analysis | not evaluable | full operating curve |
| Verdict on $H_{4.1}$ | unsupported | **refuted** |
| Overall RQ4 verdict | inconclusive | **split**: ordinal model refuted, retrieval succeeds |

The refutation is a stronger and more useful result than the original
non-finding, and the retrieval baseline beating the mode is a positive result the
paper did not previously have.

# Part 3 — Analyses added during review

## 3.1 Portfolio-composition sensitivity

A reviewer would ask whether the headline ratio is carried by the weakest learner
in the portfolio. It partly is, and the paper now says so.

In [ ]:
specs = [("as published (k=10)", []),
         ("- decision tree (k=9)", ["A2_DecTree"]),
         ("- decision tree, kNN (k=8)", ["A2_DecTree","A11_kNN"]),
         ("- decision tree, kNN, SVM (k=7)", ["A2_DecTree","A11_kNN","A5_SVM_SVR"])]
rows = []
for name, drop in specs:
    sub = ok[~ok.algo_id.isin(list(drop)+["A10_TabPFN"])]
    pp, pa = prep_pivot(sub), algo_pivot(sub, exclude=())
    d, lo, hi, n, _ = bootstrap_delta(pp, pa)
    ds = sorted(set(pp.dropna().index) & set(pa.dropna().index))
    w_p = kendall_w(pp.loc[ds])[0]; w_a = kendall_w(pa.loc[ds])[0]
    rows.append(dict(specification=name, k=pa.shape[1], eps2_prep=w_p,
                     eps2_alg=w_a, ratio=w_a/w_p, delta=d, ci_lo=lo, ci_hi=hi, N=n))
portfolio = pd.DataFrame(rows).round(4)
display(portfolio)

## 3.2 Metric artefact at the class-weighting rung

Level $P_{8\mathrm{b}}$ reweights classes; ROC-AUC is threshold-free and cannot
register that. The reported null for this rung is a property of the metric, not
of the transformation.

In [ ]:
cls = ok[ok.task=="classification"]
for label, sub in [("IR >= 5 (weighting active)", cls[cls.IR>=5]),
                   ("all classification", cls)]:
    for name, col in [("ROC-AUC (primary)","fold_mean"), ("balanced accuracy","bacc_mean")]:
        p = sub.groupby(["dataset","pipeline"])[col].mean().unstack("pipeline")
        c = p[["P8a","P8b"]].dropna()
        w, pv = wilcoxon(c.P8b, c.P8a, alternative="greater")
        print(f"{label:28s} {name:20s} N={len(c):3d} meanD={(c.P8b-c.P8a).mean():+.4f} p={pv:.4f}")

## 3.3 Diagnosis of the three empty datasets

Three of 95 datasets produced no usable cell. Both causes are properties of the
harness rather than of the data, and both are recoverable.

In [ ]:
dead = [d for d, g in raw.groupby("dataset") if g.fold_mean.notna().sum() == 0]
for d_ in dead:
    m = meta.loc[d_]
    reasons = raw[raw.dataset==d_].skip_reason.dropna().str[:55].value_counts().to_dict()
    note = ""
    if m.task == "classification" and m.imbalance_ratio > 0:
        note = ("rarest class likely below the 10 CV folds -> StratifiedKFold "
                "raises in the OUTER loop, so no per-cell reason code is written")
    elif m.n_numeric <= 1:
        note = "single numeric column -> transformer could not emit a 2-D matrix"
    print(f"{d_:16s} n={m.n_samples:>6.0f} classes={m.n_classes:>2.0f} "
          f"IR={m.imbalance_ratio:>7.1f} miss={m.miss_rate_pct:>5.1f}%  num/cat={m.n_numeric:.0f}/{m.n_categorical:.0f}")
    print(f"                 reasons={reasons}")
    print(f"                 -> {note}\n")
print("Remedy: merge rare classes (as already done for covertype and yeast),")
print("or reduce F for these datasets; for socmob, guard the single-column branch.")

# Part 4 — Reproduce the manuscript tables

## 4.1 Execution telemetry and effective sample

In [ ]:
miss = raw[raw.fold_mean.isna()]
tab = {
 "valid":                    len(ok),
 "out of scope (A10)":       (miss.skip_reason == "tabpfn_scope_exceeded").sum(),
 "dataset-level failure":    miss[miss.dataset.isin(
                                 [d for d,g in raw.groupby("dataset")
                                  if g.fold_mean.notna().sum()==0])].shape[0],
 "timeout":                  miss.skip_reason.fillna("").str.contains("Timeout").sum(),
}
tab["structural / other"] = len(raw) - sum(tab.values())
print(pd.Series(tab).to_frame("cells").assign(pct=lambda d: (100*d.cells/len(raw)).round(2)))

contributing = ok.dataset.nunique()
full_ladder  = prep_pivot(ok).dropna().shape[0]
full_algo    = algo_pivot(ok).dropna().shape[0]
both = len(set(prep_pivot(ok).dropna().index) & set(algo_pivot(ok).dropna().index))
print(f"\ncontributing {contributing} | full ladder {full_ladder} | "
      f"full algo {full_algo} | both {both}")

## 4.2 Marginal performance across the ladder

In [ ]:
rows = []
piv = prep_pivot(ok).dropna()
ranks = piv.rank(axis=1, ascending=False).mean()
for L in LEVELS:
    cls = ok[(ok.pipeline==L) & (ok.task=="classification")]
    reg = ok[(ok.pipeline==L) & (ok.task=="regression")]
    rows.append(dict(level=L, cells=(ok.pipeline==L).sum(),
                     roc_auc=cls.fold_mean.mean(), bal_acc=cls.bacc_mean.mean(),
                     r2=reg.fold_mean.mean(), mean_rank=ranks[L]))
marginal = pd.DataFrame(rows).round(4)
display(marginal)

## 4.3 Omnibus tests and effect sizes

In [ ]:
wp, chi_p, n_p, k_p = kendall_w(prep_pivot(ok))
wa, chi_a, n_a, k_a = kendall_w(algo_pivot(ok))
print(f"preprocessing: W={wp:.4f}  chi2={chi_p:.3f}  F_ID={iman_davenport(chi_p,n_p,k_p):.3f}  N={n_p} k={k_p}")
print(f"algorithm    : W={wa:.4f}  chi2={chi_a:.3f}  F_ID={iman_davenport(chi_a,n_a,k_a):.3f}  N={n_a} k={k_a}")
print(f"ratio        : {wa/wp:.2f}x")

d, lo, hi, n, p_pos = bootstrap_delta(prep_pivot(ok), algo_pivot(ok))
print(f"\nDelta eps^2 = {d:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   N={n}")
print(f"one-sided P(Delta > 0) = {p_pos:.4f}")

> **Note on two slightly different $\Delta\varepsilon^2$ values.** The released
> artefact reports $-0.3403$, which is the difference of two effect sizes each
> computed on its own maximal sample ($N=88$ for preprocessing, $N=91$ for the
> algorithm factor), with the confidence interval bootstrapped on the 87 datasets
> common to both. Recomputing *both* effect sizes on those 87 datasets gives
> $-0.3348$. The manuscript reports the released value in its omnibus table and
> the common-sample value in the portfolio-sensitivity table, and the captions
> state which sample each uses. The gap is 0.0055, well inside the interval, and
> changes no conclusion.

## 4.4 Localisation and multiplicity control

In [ ]:
piv = prep_pivot(ok)
tests, names = [], []
for lo_, hi_ in zip(LEVELS[:-1], LEVELS[1:]):
    c = piv[[lo_, hi_]].dropna()
    w, p = wilcoxon(c[hi_], c[lo_], alternative="greater")
    tests.append(p); names.append(f"{lo_} -> {hi_}")
    print(f"{lo_:>3} -> {hi_:<3} N={len(c):3d}  W={w:6.0f}  meanD={(c[hi_]-c[lo_]).mean():+.4f}  p={p:.4g}")

c = piv[["P1","P8b"]].dropna()
w, p = wilcoxon(c.P8b, c.P1, alternative="greater")
print(f"\nterminal P1 -> P8b: N={len(c)} W={w:.0f} meanD={(c.P8b-c.P1).mean():+.4f} p={p:.3g}")

In [ ]:
# moderators + high-missingness subgroup complete the 13-test secondary pool
mod = []
per_ds = {}
for d_, g in ok.groupby("dataset"):
    p = g.groupby(["algo_id","pipeline"])["fold_mean"].mean().unstack("pipeline")
    p = p.reindex(columns=LEVELS).dropna()
    if len(p) >= 2: per_ds[d_] = kendall_w(p)[0]
eff = pd.Series(per_ds, name="eps2")
X = meta.loc[eff.index].assign(cat_density=lambda d: d.n_categorical/d.n_features.clip(lower=1),
                               log10_n=lambda d: np.log10(d.n_samples))
for col, tail in [("imbalance_ratio","one"),("cat_density","two"),
                  ("log10_n","two"),("miss_rate_pct","two")]:
    rho, p2 = spearmanr(eff.values, X[col].values)
    p = p2/2 if tail=="one" else p2
    tests.append(p); names.append(f"Spearman {col}")
    print(f"{col:>18}  rho={rho:+.3f}  p={p:.4f}")

hm = X.index[X.miss_rate_pct > 5]
c = piv.loc[piv.index.isin(hm), ["P3","P4"]].dropna()
w, p = wilcoxon(c.P4, c.P3, alternative="greater")
tests.append(p); names.append("high-missingness P3->P4")

for method in ["holm","fdr_bh"]:
    rej, q, _, _ = multipletests(tests, alpha=.05, method=method)
    print(f"\n{method}: {sum(rej)} of {len(tests)} significant -> "
          f"{[n for n,r in zip(names,rej) if r]}")

## 4.5 Per-algorithm sensitivity

In [ ]:
rows = []
for a, g in ok.groupby("algo_id"):
    ws = []
    for d_, gg in g.groupby("dataset"):
        s = gg.set_index("pipeline")["fold_mean"].reindex(LEVELS)
        if s.notna().sum() == K:
            ws.append(kendall_w(pd.DataFrame([s.values], columns=LEVELS))[0])
    p = g.groupby("pipeline")["fold_mean"].mean()
    rows.append(dict(algo=a, n_datasets=g.dataset.nunique(),
                     P1=p.get("P1"), P4=p.get("P4"), P8b=p.get("P8b"),
                     delta=p.get("P8b",np.nan)-p.get("P1",np.nan)))
sens = pd.DataFrame(rows).sort_values("delta", ascending=False).round(4)
display(sens)

groups = []
for a, g in ok.groupby("algo_id"):
    p = g.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")
    c = p[["P1","P8b"]].dropna()
    if len(c) >= 2: groups.append((c.P8b - c.P1).values)
F, p = levene(*groups)
print(f"\nLevene across {len(groups)} families: F={F:.4f}  p={p:.3g}")

## 4.6 Minimal Sufficient Preprocessing

In [ ]:
def msp_labels(df, tau=TAU):
    """Earliest ladder level within `tau` of the best available level."""
    out = {}
    for (d_, a), g in df.groupby(["dataset","algo_id"]):
        s = g.set_index("pipeline")["fold_mean"].reindex(LEVELS)
        if s.notna().sum() == 0: continue
        out[(d_, a)] = int(np.argmax(((s >= s.max()-tau) & s.notna()).values))
    return out

lab = msp_labels(ok)
rows = []
for a in sorted(ok.algo_id.unique()):
    vals = {LEVELS[v] for (d_, aa), v in lab.items() if aa == a}
    per_tau = {}
    for t in (0.005, 0.01, 0.02):
        l = msp_labels(ok, t)
        ks = [v for (d_, aa), v in l.items() if aa == a]
        per_tau[t] = LEVELS[pd.Series(ks).value_counts().sort_index().idxmax()]
    rows.append(dict(algo=a, **{f"tau={t}": v for t, v in per_tau.items()},
                     stable=len(set(per_tau.values()))==1))
msp = pd.DataFrame(rows)
display(msp)
print("cross-tolerance agreement:", f"{msp.stable.sum()}/{len(msp)}")

## 4.7 Export

In [ ]:
for name, df in [("marginal", marginal), ("portfolio_sensitivity", portfolio),
                 ("per_algorithm_sensitivity", sens), ("msp_policy", msp),
                 ("pmsp_lodo_summary", summary.reset_index()),
                 ("pmsp_lodo_predictions", pred)]:
    path = OUT/f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"wrote {path}  ({len(df)} rows)")

# Part 5 — Verification

Everything above is re-derivation. This part is adjudication: 273 assertions that
the manuscript and the data agree.

Three layers, because a data-only check is not enough. The one real error found
during writing was a correct analysis transcribed incorrectly into the text —
invisible to layer A, caught by layer C.

| Layer | Question | Checks |
|---|---|---:|
| A | Does the data reproduce each reported number? | 234 |
| B | Does that number appear in `manuscript.tex`? | 28 |
| C | Is each derived ratio consistent with its own inputs? | 11 |

## 5.1 Check framework and manuscript source

In [ ]:
CHECKS = []

def check(section, name, computed, expected, tol=5e-4, note=""):
    """Register one data->value check."""
    ok = (computed is not None
          and not (isinstance(computed, float) and np.isnan(computed))
          and abs(float(computed) - float(expected)) <= tol)
    CHECKS.append(dict(layer="A", section=section, quantity=name,
                       computed=round(float(computed), 6) if computed is not None else None,
                       expected=float(expected), tol=tol, pass_=bool(ok), note=note))
    return ok

# Layer B compares reported values against the manuscript source. In a
# code-only repository the .tex is absent, so those checks are skipped rather
# than failed: their absence is a property of the repository, not of the work.
_tex_path = DATA / "manuscript.tex"
TEX_AVAILABLE = _tex_path.exists()
TEX = _tex_path.read_text() if TEX_AVAILABLE else ""
if not TEX_AVAILABLE:
    print("manuscript.tex not found -> Layer B (manuscript text) will be "
          "SKIPPED.\n   Layers A (data) and C (arithmetic) still run in full.\n"
          "   To enable Layer B, place manuscript.tex in this directory.")

def in_tex(section, name, literal, expect_present=True):
    """Register one value->manuscript check: does the string occur in the .tex?"""
    if not TEX_AVAILABLE:
        return None                      # skipped, not failed
    n = TEX.count(str(literal))
    ok = (n > 0) if expect_present else (n == 0)
    CHECKS.append(dict(layer="B", section=section, quantity=name, computed=n,
                       expected="present" if expect_present else "absent",
                       tol=None, pass_=bool(ok), note=str(literal)))
    return ok

def arith(section, name, computed, claimed, tol):
    """Register one internal-arithmetic check (ratios, percentages)."""
    ok = abs(computed - claimed) <= tol
    CHECKS.append(dict(layer="C", section=section, quantity=name,
                       computed=round(computed, 4), expected=claimed, tol=tol,
                       pass_=bool(ok), note=""))
    return ok

def kendall_w(piv):
    piv = piv.dropna(); n, k = piv.shape
    r = piv.rank(axis=1, ascending=False).values
    chi2_f = 12*n/(k*(k+1)) * np.sum((r.mean(0) - (k+1)/2)**2)
    return chi2_f/(n*(k-1)), chi2_f, n, k

def iman_davenport(chi2_f, n, k):
    d = n*(k-1) - chi2_f
    return np.inf if d <= 0 else (n-1)*chi2_f/d

print("framework ready")

In [ ]:
raw  = pd.read_csv(DATA/"raw_results.csv")
meta = pd.read_csv(DATA/"table1_dataset_characteristics.csv").set_index("dataset")
ok   = raw[raw.fold_mean.notna()].copy()
ok["task"] = ok.dataset.map(meta.task)
ok["IR"]   = ok.dataset.map(meta.imbalance_ratio)

prep = ok.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")[LEVELS]
algo = (ok[ok.algo_id != "A10_TabPFN"]
          .groupby(["dataset","algo_id"])["fold_mean"].mean().unstack("algo_id"))
print(f"loaded: {len(raw):,} scheduled cells, {len(ok):,} valid")

## 5.2 Layer A — the data reproduces the reported values

In [ ]:
check("4.1","scheduled cells", len(raw), 9405, 0)
check("4.1","valid cells", len(ok), 8610, 0)
check("4.1","valid share (%)", 100*len(ok)/len(raw), 91.55, 0.01)
check("4.1","datasets loaded", raw.dataset.nunique(), 95, 0)
check("4.1","datasets contributing", ok.dataset.nunique(), 92, 0)
check("4.1","full ladder coverage", prep.dropna().shape[0], 88, 0)
check("4.1","full Stage-A coverage", algo.dropna().shape[0], 91, 0)
check("4.1","both complete", len(set(prep.dropna().index) & set(algo.dropna().index)), 87, 0)
check("4.1","classification datasets", (meta.task=="classification").sum(), 62, 0)
check("4.1","regression datasets", (meta.task=="regression").sum(), 33, 0)
check("4.1","total observations", meta.n_samples.sum(), 1_234_570, 0)
check("4.1","datasets with categoricals", (meta.n_categorical>0).sum(), 24, 0)
check("4.1","datasets with miss > 5%", (meta.miss_rate_pct>5).sum(), 4, 0)
check("4.1","datasets with IR >= 5", (meta.imbalance_ratio>=5).sum(), 15, 0)
check("4.1","multiclass datasets", (meta.n_classes>=3).sum(), 26, 0)
dead = [d for d,g in raw.groupby("dataset") if g.fold_mean.notna().sum()==0]
check("4.1","datasets with zero valid cells", len(dead), 3, 0, note=", ".join(sorted(dead)))
print(f"registered {len(CHECKS)} checks")

In [ ]:
EXPECT = {  # level: (roc_auc, r2, mean_rank)
 "P1":(0.8776,0.5691,6.489), "P2":(0.8903,0.5719,5.898), "P3":(0.8899,0.5762,5.909),
 "P4":(0.9046,0.6225,3.926), "P5":(0.9044,0.6225,3.892), "P6":(0.9021,0.6210,5.301),
 "P7":(0.9037,0.6179,4.318), "P8a":(0.9037,0.6175,4.562), "P8b":(0.9046,0.6175,4.705)}
ranks = prep.dropna().rank(axis=1, ascending=False).mean()
for L,(auc,r2,rk) in EXPECT.items():
    cls = ok[(ok.pipeline==L)&(ok.task=="classification")].fold_mean.mean()
    reg = ok[(ok.pipeline==L)&(ok.task=="regression")].fold_mean.mean()
    check("4.2",f"{L} ROC-AUC", cls, auc, 5e-5)
    check("4.2",f"{L} R2", reg, r2, 5e-5)
    check("4.2",f"{L} mean rank", ranks[L], rk, 5e-4)

t4 = pd.read_csv(TABLES/"table4_pivot.csv")
check("4.2","terminal gain mean", t4.delta.mean(), 0.0311, 5e-5)
check("4.2","terminal gain median", t4.delta.median(), 0.0048, 5e-5)
check("4.2","datasets with positive gain", (t4.delta>0).sum(), 61, 0)
check("4.2","mean lift over chance", t4.lift.mean(), 0.4872, 5e-5)
print(f"registered {len(CHECKS)} checks")

In [ ]:
wp, chip, np_, kp = kendall_w(prep)
wa, chia, na, ka  = kendall_w(algo)
check("4.3","W preprocessing", wp, 0.1177, 5e-5)
check("4.3","W algorithm", wa, 0.4580, 5e-5)
check("4.3","chi2 preprocessing", chip, 82.877, 5e-3)
check("4.3","chi2 algorithm", chia, 375.081, 5e-3)
check("4.3","F_ID preprocessing", iman_davenport(chip,np_,kp), 11.609, 5e-3)
check("4.3","F_ID algorithm", iman_davenport(chia,na,ka), 76.044, 5e-3)
check("4.3","Delta (own-sample difference)", wp-wa, -0.3403, 5e-5)
arith("4.3","ratio W_alg / W_prep", wa/wp, 3.9, 0.05)

# task strata
for task,(p_e,a_e) in {"classification":(0.0975,0.4772),"regression":(0.1800,0.4833)}.items():
    sub = ok[ok.task==task]
    pw = kendall_w(sub.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")[LEVELS])[0]
    aw = kendall_w(sub[sub.algo_id!="A10_TabPFN"].groupby(["dataset","algo_id"])["fold_mean"]
                     .mean().unstack("algo_id"))[0]
    check("4.3",f"W prep ({task})", pw, p_e, 5e-5)
    check("4.3",f"W alg ({task})", aw, a_e, 5e-5)
print(f"registered {len(CHECKS)} checks")

In [ ]:
SPECS = [([], 10, 0.1271, 0.4596, -0.3326, 3.6),
         (["A2_DecTree"], 9, 0.1260, 0.3591, -0.2331, 2.9),
         (["A2_DecTree","A11_kNN"], 8, 0.1489, 0.2227, -0.0738, 1.5),
         (["A2_DecTree","A11_kNN","A5_SVM_SVR"], 7, 0.0630, 0.1341, -0.0711, None)]
for drop, k_exp, wp_e, wa_e, d_e, ratio_e in SPECS:
    sub = ok[~ok.algo_id.isin(list(drop)+["A10_TabPFN"])]
    pp = sub.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")[LEVELS]
    pa = sub.groupby(["dataset","algo_id"])["fold_mean"].mean().unstack("algo_id")
    ds = sorted(set(pp.dropna().index) & set(pa.dropna().index))
    w_p, w_a = kendall_w(pp.loc[ds])[0], kendall_w(pa.loc[ds])[0]
    tag = f"k={k_exp}"
    check("4.3",f"{tag} n algorithms", pa.shape[1], k_exp, 0)
    check("4.3",f"{tag} W prep", w_p, wp_e, 5e-5)
    check("4.3",f"{tag} W alg", w_a, wa_e, 5e-5)
    check("4.3",f"{tag} Delta", w_p-w_a, d_e, 5e-5)
    check("4.3",f"{tag} common N", len(ds), 87, 0)
    if ratio_e is not None:
        arith("4.3",f"{tag} ratio", w_a/w_p, ratio_e, 0.06)
print(f"registered {len(CHECKS)} checks")

In [ ]:
PAIRS = {("P1","P2"):(88,1979,0.0056), ("P2","P3"):(92,1686,0.0004),
         ("P3","P4"):(92,3596,0.0257), ("P4","P5"):(92,1640,-0.0001),
         ("P5","P6"):(92,1334,-0.0020), ("P6","P7"):(92,2679,-0.0001),
         ("P7","P8a"):(92,1343,-0.0001), ("P8a","P8b"):(92,1622,0.0006)}
for (lo,hi),(n_e,w_e,d_e) in PAIRS.items():
    c = prep[[lo,hi]].dropna()
    w,_ = wilcoxon(c[hi], c[lo], alternative="greater")
    check("4.4",f"{lo}->{hi} N", len(c), n_e, 0)
    check("4.4",f"{lo}->{hi} W stat", w, w_e, 0.5)
    check("4.4",f"{lo}->{hi} mean diff", (c[hi]-c[lo]).mean(), d_e, 5e-5)

c = prep[["P1","P8b"]].dropna()
w,_ = wilcoxon(c.P8b, c.P1, alternative="greater")
check("4.4","terminal W stat", w, 3105, 0.5)
check("4.4","critical difference k=9", 3.102*np.sqrt(9*10/(6*88)), 1.281, 5e-4)
check("4.4","critical difference k=10", 3.164*np.sqrt(10*11/(6*91)), 1.420, 5e-4)
print(f"registered {len(CHECKS)} checks")

In [ ]:
cls = ok[ok.task=="classification"]
CASES = [("IR>=5","fold_mean",13,0.0042), ("IR>=5","bacc_mean",13,0.0561),
         ("all","fold_mean",60,0.0009),   ("all","bacc_mean",60,0.0121)]
for scope, col, n_e, d_e in CASES:
    sub = cls[cls.IR>=5] if scope=="IR>=5" else cls
    p = sub.groupby(["dataset","pipeline"])[col].mean().unstack("pipeline")
    c = p[["P8a","P8b"]].dropna()
    metric = "ROC-AUC" if col=="fold_mean" else "bal.acc"
    check("4.4",f"P8a->P8b {metric} ({scope}) N", len(c), n_e, 0)
    check("4.4",f"P8a->P8b {metric} ({scope}) diff", (c.P8b-c.P8a).mean(), d_e, 5e-5)
arith("4.4","metric factor (bacc / auc on IR>=5)", 0.0561/0.0042, 13, 0.4)
print(f"registered {len(CHECKS)} checks")

In [ ]:
GAINS = {"A5_SVM_SVR":0.1005,"A11_kNN":0.0658,"A1_Ridge_LogReg":0.1390,"A9_MLP":0.0157,
         "A7_LightGBM":0.0167,"A8_CatBoost":0.0130,"A4_GradBoosting_SKL":0.0151,
         "A6_XGBoost":0.0130,"A3_RandForest":0.0144,"A10_TabPFN":0.0182,"A2_DecTree":0.0104}
for a, g_e in GAINS.items():
    p = ok[ok.algo_id==a].groupby("pipeline")["fold_mean"].mean()
    check("4.5",f"{a} terminal gain", p["P8b"]-p["P1"], g_e, 5e-5)

groups = []
for a,g in ok.groupby("algo_id"):
    p = g.groupby(["dataset","pipeline"])["fold_mean"].mean().unstack("pipeline")
    c = p[["P1","P8b"]].dropna()
    if len(c)>=2: groups.append((c.P8b-c.P1).values)
F,_ = levene(*groups)
check("4.5","Levene F", F, 3.9268, 5e-4)
check("4.5","Levene groups", len(groups), 11, 0)

rk = algo.dropna().rank(axis=1, ascending=False)
check("4.5","decision tree mean rank", rk["A2_DecTree"].mean(), 9.000, 5e-4)
check("4.5","decision tree strictly last", int((rk["A2_DecTree"]==10).sum()), 56, 0)
check("4.5","LightGBM mean rank", rk["A7_LightGBM"].mean(), 3.593, 5e-4)

In [ ]:
# The manuscript's moderator analysis uses the released moderator_data.csv, whose
# per-dataset effect is computed over each dataset's available cells. Recomputing
# it here with a stricter full-ladder requirement would give a different sample
# (N=88 rather than 92) and different correlations, so we verify against the
# artefact the manuscript actually used.
mod = pd.read_csv(DATA/"moderator_data.csv")
check("4.6","moderator N", len(mod), 92, 0)
check("4.6","per-dataset effect median", mod.eps2_prep.median(), 0.0933, 5e-5)
check("4.6","per-dataset effect max", mod.eps2_prep.max(), 0.5845, 5e-5)
check("4.6","per-dataset effect min", mod.eps2_prep.min(), 0.0022, 5e-5)
for col, rho_e in [("cat_density",0.313),("miss_rate",0.194),
                   ("log10_n",0.095),("IR_max_min",-0.058)]:
    rho,_ = spearmanr(mod.eps2_prep.values, mod[col].values)
    check("4.6",f"Spearman {col}", rho, rho_e, 5e-4)

split = mod.assign(hascat=mod.cat_density>0).groupby("hascat").eps2_prep.agg(["mean","median","size"])
check("4.6","effect | categorical present", split.loc[True,"mean"], 0.2376, 5e-5)
check("4.6","effect | numeric only", split.loc[False,"mean"], 0.0991, 5e-5)
check("4.6","datasets with categoricals (moderator sample)", split.loc[True,"size"], 22, 0)
arith("4.6","categorical/numeric ratio", 0.2376/0.0991, 2.4, 0.01)
print(f"registered {len(CHECKS)} checks")

In [ ]:
def msp_labels(df, tau):
    out = {}
    for (d,a),g in df.groupby(["dataset","algo_id"]):
        s = g.set_index("pipeline")["fold_mean"].reindex(LEVELS)
        if s.notna().sum()==0: continue
        out[(d,a)] = int(np.argmax(((s >= s.max()-tau) & s.notna()).values))
    return out

modes = {}
for tau in (0.005, 0.01, 0.02):
    lab = msp_labels(ok, tau)
    for a in sorted(ok.algo_id.unique()):
        ks = [v for (d,aa),v in lab.items() if aa==a]
        modes[(a,tau)] = LEVELS[pd.Series(ks).value_counts().sort_index().idxmax()]

n_p1 = sum(1 for a in ok.algo_id.unique() if modes[(a,0.01)]=="P1")
check("4.7","families with MSP = P1 at tau=1%", n_p1, 11, 0)
for a, exp in [("A5_SVM_SVR","P4"), ("A11_kNN","P4"), ("A1_Ridge_LogReg","P1")]:
    got = modes[(a,0.005)]
    CHECKS.append(dict(layer="A", section="4.7", quantity=f"{a} MSP at tau=0.5%",
                       computed=got, expected=exp, tol=None, pass_=got==exp, note=""))
stable = sum(1 for a in ok.algo_id.unique()
             if len({modes[(a,t)] for t in (0.005,0.01,0.02)})==1)
check("4.7","cross-tolerance stable families", stable, 9, 0)
print(f"registered {len(CHECKS)} checks")

## 5.3 Predictive-MSP checks

The check on finite confidences is load-bearing: zero there means the Part 1 patch did not take effect and the silent fallback has returned.

In [ ]:
from mord import LogisticAT
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

SENS = {"A2_DecTree":0,"A3_RandForest":0,"A4_GradBoosting_SKL":0,"A6_XGBoost":0,
        "A7_LightGBM":0,"A8_CatBoost":0,"A10_TabPFN":1,"A1_Ridge_LogReg":2,
        "A9_MLP":3,"A5_SVM_SVR":4,"A11_kNN":4}
FEATS = ["log10_n","p","log10_np","miss","cat_density","has_hicard","ir","n_classes","is_cls","sens"]

def build(tau):
    rows=[]
    for (d,a),g in ok.groupby(["dataset","algo_id"]):
        s = g.set_index("pipeline")["fold_mean"].reindex(LEVELS)
        if s.notna().sum()==0: continue
        m = meta.loc[d]
        rows.append(dict(dataset=d, algo_id=a,
            true_idx=int(np.argmax(((s>=s.max()-tau)&s.notna()).values)),
            log10_n=np.log10(m.n_samples), p=m.n_features,
            log10_np=np.log10(m.n_samples/max(m.n_features,1)), miss=m.miss_rate_pct,
            cat_density=m.n_categorical/max(m.n_features,1),
            has_hicard=float(m.n_high_card_cat>0), ir=m.imbalance_ratio,
            n_classes=m.n_classes, is_cls=float(m.task=="classification"), sens=SENS[a],
            best=s.max(), **{f"s_{L}": s[L] for L in LEVELS}))
    return pd.DataFrame(rows)

def gap(row,p):
    for j in range(p,K):
        if pd.notna(row[f"s_{LEVELS[j]}"]): return float(row.best-row[f"s_{LEVELS[j]}"])
    return np.nan

def lodo(inst, seed):
    inst = inst.copy()
    inst["regime"] = list(zip(inst.log10_n>=np.log10(5000), inst.miss>5, inst.ir>=5))
    out=[]
    for held in sorted(inst.dataset.unique()):
        tr, te = inst[inst.dataset!=held], inst[inst.dataset==held]
        if tr.dataset.nunique()<3: continue
        sc = StandardScaler().fit(tr[FEATS].values)
        Xtr, Xte, ytr = sc.transform(tr[FEATS]), sc.transform(te[FEATS]), tr.true_idx.values
        classes = np.unique(ytr); enc = {c:i for i,c in enumerate(classes)}
        mdl = LogisticAT(alpha=1.0).fit(Xtr, np.array([enc[v] for v in ytr]))
        pred = classes[np.clip(mdl.predict(Xte).astype(int), 0, len(classes)-1)]
        conf = mdl.predict_proba(Xte).max(axis=1)
        P = {"P-MSP_meta":(pred,conf), "B0_global":(np.full(len(te),int(np.median(ytr))),None),
             "B4_oracle":(te.true_idx.values,None)}
        m1 = tr.groupby("algo_id").true_idx.agg(lambda s:int(s.value_counts().sort_index().idxmax()))
        P["B1_algo_mode"]=(te.algo_id.map(m1).fillna(int(np.median(ytr))).astype(int).values,None)
        m2 = tr.groupby("regime").true_idx.agg(lambda s:int(s.value_counts().sort_index().idxmax()))
        P["B2_regime"]=(te.regime.map(m2).fillna(int(np.median(ytr))).astype(int).values,None)
        nn = NearestNeighbors(n_neighbors=min(25,len(tr))).fit(Xtr)
        idx = nn.kneighbors(Xte, return_distance=False)
        P["B3_meta_knn"]=(np.array([int(pd.Series(ytr[i]).value_counts().sort_index().idxmax())
                                    for i in idx]),None)
        for meth,(pr,cf) in P.items():
            for j,(_,row) in enumerate(te.iterrows()):
                p=int(pr[j])
                out.append(dict(seed=seed, method=meth, dataset=held, algo_id=row.algo_id,
                    true_idx=int(row.true_idx), pred_idx=p, exact=int(p==row.true_idx),
                    off_by_one=int(abs(p-row.true_idx)<=1), abs_err=abs(p-row.true_idx),
                    confidence=np.nan if cf is None else float(cf[j]),
                    perf_gap=gap(row,p), fit_fraction=(p+1)/K))
    return pd.DataFrame(out)

inst = build(TAU)
pred = pd.concat([lodo(inst,s) for s in SEEDS], ignore_index=True)
print(f"instances {len(inst)} | predictions {len(pred)}")

In [ ]:
check("4.8","instances", len(inst), 963, 0)
check("4.8","instances x seeds per method", (pred.method=="B1_algo_mode").sum(), 2889, 0)
check("4.8","majority-class base rate", inst.true_idx.value_counts(normalize=True).max(), 0.7092, 5e-5)
check("4.8","P1 label share (%)", 100*(inst.true_idx==0).mean(), 70.92, 0.01)

g = pred.groupby("method").agg(exact=("exact","mean"), off1=("off_by_one","mean"),
                               mae=("abs_err","mean"), gap=("perf_gap","mean"),
                               fit=("fit_fraction","mean"))
EXP = {"B0_global":(0.7092,0.8505,0.756,0.0374,0.111),
       "B1_algo_mode":(0.7092,0.8505,0.756,0.0374,0.111),
       "B2_regime":(0.7009,0.8505,0.761,0.0374,0.112),
       "B3_meta_knn":(0.7799,0.8598,0.633,0.0315,0.141),
       "B4_oracle":(1.0000,1.0000,0.000,0.0010,0.195),
       "P-MSP_meta":(0.6490,0.8318,0.813,0.0336,0.160)}
for m,(e,o,ma,gp,ft) in EXP.items():
    check("4.8",f"{m} exact", g.loc[m,"exact"], e, 5e-5)
    check("4.8",f"{m} off-by-one", g.loc[m,"off1"], o, 5e-5)
    check("4.8",f"{m} MAE", g.loc[m,"mae"], ma, 5e-4)
    check("4.8",f"{m} gap", g.loc[m,"gap"], gp, 5e-5)
    check("4.8",f"{m} fit fraction", g.loc[m,"fit"], ft, 5e-4)

# the fix works only if confidences exist
n_conf = pred[pred.method=="P-MSP_meta"].confidence.notna().sum()
check("4.8","finite confidence values", n_conf, 2889, 0,
      note="zero here means the label-encoding fix did not take effect")

In [ ]:
piv = pred.pivot_table(index=["seed","dataset","algo_id"], columns="method", values="exact")
def mcn(a,b):
    n01=int(((piv[a]==1)&(piv[b]==0)).sum()); n10=int(((piv[a]==0)&(piv[b]==1)).sum())
    stat=0.0 if n01+n10==0 else (abs(n01-n10)-1)**2/(n01+n10)
    return n01,n10,stat
MCN = {("P-MSP_meta","B1_algo_mode"):(96,270,81.77),
       ("P-MSP_meta","B3_meta_knn"):(147,525,211.50),
       ("B3_meta_knn","B1_algo_mode"):(360,156,79.86)}
for (a,b),(w_e,l_e,s_e) in MCN.items():
    w,l,s = mcn(a,b)
    check("4.8",f"McNemar {a} vs {b} wins", w, w_e, 0)
    check("4.8",f"McNemar {a} vs {b} losses", l, l_e, 0)
    check("4.8",f"McNemar {a} vs {b} chi2", s, s_e, 0.01)

m = pred[(pred.method=="P-MSP_meta") & pred.confidence.notna()]
for t, cov_e, acc_e in [(0.30,0.975,0.657),(0.70,0.628,0.861),(0.90,0.162,0.910)]:
    ans = (m.confidence>=t).values
    check("4.8",f"abstention t={t} coverage", ans.mean(), cov_e, 5e-4)
    check("4.8",f"abstention t={t} accuracy on covered", m.exact[ans].mean(), acc_e, 5e-4)
print(f"registered {len(CHECKS)} checks")

In [ ]:
# tolerance sweep and reliable-label subset
SWEEP = {0.005:(0.6532,0.5929,0.6625,0.7456), 0.01:(0.7092,0.6490,0.7092,0.7799),
         0.02:(0.7570,0.7103,0.7570,0.8276)}
for tau,(base_e,p_e,b1_e,b3_e) in SWEEP.items():
    i2 = build(tau); p2 = pd.concat([lodo(i2,s) for s in SEEDS], ignore_index=True)
    gg = p2.groupby("method").exact.mean()
    check("4.8",f"tau={tau} base rate", i2.true_idx.value_counts(normalize=True).max(), base_e, 5e-5)
    check("4.8",f"tau={tau} P-MSP", gg["P-MSP_meta"], p_e, 5e-5)
    check("4.8",f"tau={tau} B1", gg["B1_algo_mode"], b1_e, 5e-5)
    check("4.8",f"tau={tau} B3", gg["B3_meta_knn"], b3_e, 5e-5)

def labels_of(fname):
    r = pd.read_csv(DATA/fname); r = r[r.fold_mean.notna()]; o={}
    for (d,a),g in r.groupby(["dataset","algo_id"]):
        s = g.set_index("pipeline")["fold_mean"].reindex(LEVELS)
        if s.notna().sum()==0: continue
        o[(d,a)] = int(np.argmax(((s>=s.max()-TAU)&s.notna()).values))
    return o
L3 = [labels_of(f) for f in ["raw_results.csv","results_seed7.csv","results_seed13.csv"]]
common = set(L3[0]) & set(L3[1]) & set(L3[2])
stable = {k for k in common if L3[0][k]==L3[1][k]==L3[2][k]}
check("4.8","instances common to 3 seeds", len(common), 962, 0)
check("4.8","seed-consistent labels", len(stable), 891, 0)
arith("4.8","seed-consistent share (%)", 100*len(stable)/len(common), 92.6, 0.05)

key = pd.Series(list(zip(pred.dataset,pred.algo_id)), index=pred.index)
sub = pred[key.isin(stable)]
gs = sub.groupby("method").agg(e=("exact","mean"), gp=("perf_gap","mean"))
for m,(e_e,g_e) in {"B1_algo_mode":(0.7407,0.0362),"B3_meta_knn":(0.8182,0.0300),
                    "P-MSP_meta":(0.6824,0.0328)}.items():
    check("4.8",f"reliable subset {m} exact", gs.loc[m,"e"], e_e, 5e-5)
    check("4.8",f"reliable subset {m} gap", gs.loc[m,"gp"], g_e, 5e-5)
print(f"registered {len(CHECKS)} checks")

In [ ]:
ms = pd.read_csv(DATA/"multiseed_delta_eps2.csv").set_index("seed")
for sd,(p_e,a_e,d_e) in {"7":(0.1315,0.4576,-0.3261),"13":(0.1509,0.4636,-0.3127)}.items():
    check("4.9",f"seed {sd} W prep", ms.loc[sd,"eps2_prep"], p_e, 5e-5)
    check("4.9",f"seed {sd} W alg", ms.loc[sd,"eps2_algo_stageA"], a_e, 5e-5)
    check("4.9",f"seed {sd} Delta", ms.loc[sd,"delta_eps2"], d_e, 5e-5)
check("4.9","across-seed sd", ms.loc["SUMMARY","delta_sd_across_seeds"], 0.0138, 5e-5)

t8 = pd.read_csv(TABLES/"table8_loo_sensitivity.csv")
check("4.9","LOO refits", len(t8), 87, 0)
check("4.9","LOO min Delta", t8.delta_eps2.min(), -0.3456, 5e-5)
check("4.9","LOO max Delta", t8.delta_eps2.max(), -0.3178, 5e-5)
check("4.9","LOO sign flips", int((t8.delta_eps2>0).sum()), 0, 0)
arith("4.9","max displacement (%)", 100*(t8.delta_eps2.max()-t8.delta_eps2.min())/0.3403, 8.2, 0.05)

tu = pd.read_csv(DATA/"tuned_results.csv")
check("4.9","tuned cells", len(tu), 675, 0)
check("4.9","tuned datasets", tu.dataset.nunique(), 15, 0)
check("4.9","tuned algorithms", tu.algo_id.nunique(), 5, 0)
tp = tu.groupby(["dataset","pipeline"])["tuned_mean"].mean().unstack("pipeline")[LEVELS].dropna()
check("4.9","tuned P1 mean", tp.P1.mean(), 0.6285, 5e-5)
check("4.9","tuned P4 mean", tp.P4.mean(), 0.6984, 5e-5)
arith("4.9","tuning attenuation (%)", 100*(0.2005-0.0431)/0.2005, 78.5, 0.1)
arith("4.9","W_alg contraction (%)", 100*(0.5138-0.2916)/0.5138, 43.2, 0.1)
arith("4.9","W_prep contraction (%)", 100*(0.3133-0.2485)/0.3133, 20.7, 0.1)

v = pd.read_csv(DATA/"variance_stability_5rep.csv"); v = v[v.rep_std.notna()].copy()
v["task"] = v.dataset.map(meta.task)
stab = v.apply(lambda r: r.rep_std < (0.01 if r.task=="classification"
                                      else max(0.02, 0.05*abs(r.rep_mean))), axis=1)
check("4.9","variance cells", len(v), 666, 0)
check("4.9","stable share (%)", 100*stab.mean(), 78.5, 0.1)
mm = v.merge(raw[["dataset","algo_id","pipeline","fold_mean"]],
             on=["dataset","algo_id","pipeline"]).dropna(subset=["fold_mean"])
rho,_ = spearmanr(mm.fold_mean, mm.rep_mean)
check("4.9","single vs 5x10 Spearman", rho, 0.9973, 5e-4)
print(f"registered {len(CHECKS)} checks")

## 5.4 Layers B and C — manuscript text and internal arithmetic

In [ ]:
PRESENT = [
 ("4.1","scheduled cells","9405"), ("4.1","valid cells","8610"), ("4.1","valid share","91.55"),
 ("4.3","W prep","0.1177"), ("4.3","W alg","0.4580"), ("4.3","Delta","0.3403"),
 ("4.3","chi2 prep","82.877"), ("4.3","chi2 alg","375.081"),
 ("4.3","portfolio k=9 Delta","0.2331"), ("4.3","portfolio k=8 Delta","0.0738"),
 ("4.4","scaling step","0.0257"), ("4.4","CD k=9","1.281"), ("4.4","CD k=10","1.420"),
 ("4.4","bacc effect","0.0561"), ("4.5","SVM sensitivity","0.5552"),
 ("4.6","cat density rho","0.313"), ("4.8","base rate","0.7092"),
 ("4.8","P-MSP exact","0.6490"), ("4.8","B3 exact","0.7799"),
 ("4.8","reliable labels","891"), ("4.9","tuning attenuation","78.5"),
 ("4.9","stability share","78.5"), ("4.9","Spearman 5x10","0.997"),
]
for sec,name,lit in PRESENT: in_tex(sec,name,lit,True)

# stale values that must NOT survive anywhere
# "0.7103" legitimately appears in the tolerance sweep, so the stale-value probe
# must target the wording that carried the old result, not the bare number.
ABSENT = [("4.8","old B3 wording","meta-$k$NN (0.8359)"), ("4.8","old gap","0.0392"),
          ("4.3","erroneous ratio","factor of 1.8 once"),
          ("all","fallback wording","not evaluable"),
          ("all","old RQ4 verdict","inconclusive")]
for sec,name,lit in ABSENT: in_tex(sec,name,lit,False)
print(f"registered {len(CHECKS)} checks")

## 5.5 Report

In [ ]:
rep = pd.DataFrame(CHECKS)
failed = rep[~rep.pass_]
by_layer = rep.groupby("layer").pass_.agg(["sum","count"])
by_layer.columns = ["passed","total"]
print(by_layer.to_string(), "\n")
print(rep.groupby("section").pass_.agg(["sum","count"]).rename(
      columns={"sum":"passed","count":"total"}).to_string())
print(f"\nTOTAL: {rep.pass_.sum()} / {len(rep)} passed")
rep.to_csv("verification_report.csv", index=False)
print("wrote verification_report.csv")

if len(failed):
    print(f"\n{'='*70}\nFAILURES ({len(failed)})\n{'='*70}")
    for _,r in failed.iterrows():
        print(f"  [{r.layer}] §{r.section} {r.quantity}")
        print(f"        computed={r.computed}  expected={r.expected}  {r.note}")
else:
    print("\nAll checks passed. Every reported quantity is reproduced by the released")
    print("data and appears in the manuscript; no stale value survives.")

In [ ]:
assert rep.pass_.all(), (
    f"{(~rep.pass_).sum()} verification checks FAILED — see the list above and "
    "verification_report.csv. Do not submit until these are resolved.")
print("VERIFICATION PASSED")

---

## If something fails

| Layer | Meaning | Action |
|---|---|---|
| **A** | The data no longer produce the reported value | Check you are using the released artefacts rather than a fresh run |
| **B** | The value is right in the data but absent from the `.tex` | Transcription drift — find the sentence or table and correct it |
| **C** | A ratio disagrees with its own inputs | Arithmetic error in the text |

## Applying the patch to the original pipeline

To carry the fix back into `PreProcessing.ipynb`: replace the model block of
`Cell 8d` with `fit_ordinal_msp` from Part 1.4, then re-run that cell. Everything
downstream — the LODO tables, the abstention curve, the feature importances —
follows from it.

Re-run this notebook after any change to the manuscript or the data.